In [ ]:
!pip install --upgrade pip
!pip install torch transformers
!pip install spacy
!pip install accelerate
!pip install openai
!python -m spacy download en_core_web_trf
!pip install -U scikit-learn  

In [ ]:
import os
from getpass import getpass

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

In [ ]:
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
import openai
openai.api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B", add_prefix_space=True)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B",
    dtype=torch.float16,
    device_map="auto"
)
device = next(model.parameters()).device
device

In [ ]:
import spacy
import string
import traceback
import random
import re
from typing import Optional
nlp = spacy.load("en_core_web_trf")

In [ ]:
def is_punctuation(token):
    return all(char in ".’,!?;:-()[]{}\"'" for char in token)

def is_valid_english_token(token):
    if any(char in token for char in [":", "\\"]):
        return False
    if token == "...":
        return False
    return bool(re.fullmatch(r"[a-zA-Z0-9'\".,!?;’%$€£¥]*", token))

aux_token = ["aren", "wer", "weren", "wasn", "doesn", "didn", "couldn", "shan", "cannot", "didnt", "dont", "don", "youre", "im", "theyre"]

def in_aux_token(token):
    return token.lower() in aux_token

def is_new_word(token_str: str) -> bool:
    return token_str.startswith("Ġ") or token_str.startswith("▁") or token_str.startswith(" ")

def clean_token(token_str: str) -> str:
    return token_str.strip().lstrip("Ġ▁ ")

def get_contextual_pos(nlp, context: str, candidate: str) -> Optional[str]:
    if is_punctuation(clean_token(candidate)):
        combined_text = (context + candidate).strip()
    else:
        combined_text = (context + " " + candidate).strip()
    doc = nlp(combined_text)
    if doc:
        return doc[-1].pos_
    return None

def get_contextual_pos_append(nlp, context: str, candidate: str) -> Optional[str]:
    combined_text = (context + candidate).strip()
    doc = nlp(combined_text)
    if doc:
        return doc[-1].pos_
    return None

def get_generated_tokens(nlp, prompt: str, full_text: str):
    doc = nlp(full_text)
    return [(token.text, token.pos_) for token in doc if token.idx >= len(prompt)]

In [ ]:
def has_single_pos(combined_word):
    combined_pos = nlp(combined_word)
    pos_tags = set(token.pos_ for token in combined_pos if token.pos_ != "SPACE")
    return len(pos_tags) == 1

def check_pos_tags(nlp, context: str, candidate: str) -> set[str]:
    combined_text = (context + candidate).strip()
    doc = nlp(combined_text)
    if not doc:
        return set()
    candidate_pos_tags = set()
    last_token = doc[-1]
    if last_token.pos_ != "SPACE":
        candidate_pos_tags.add(last_token.pos_)
    return candidate_pos_tags

def starts_with_uppercase(text):
    text = text.lstrip()
    return text[0].isupper() if text else False

def is_all_letters(s):
    return re.fullmatch(r"[A-Za-z]+", s) is not None

def is_all_punct(s):
    return re.fullmatch(r"[^\w\s]+", s) is not None

In [ ]:
# adjusted after the initial findings for POS tag to group assignment
def randomize_pos_groups_general():
    block = ["DET", "PRON", "ADJ"]
    others = [
        "NOUN", "VERB", "AUX", "PROPN",
        "ADV", "SCONJ", "PUNCT", "ADP",
        "CCONJ", "INTJ", "PART", "NUM", "SYM"
    ]
    atoms = [block] + [[tag] for tag in others]
    total_tags = len(block) + len(others)
    while True:
        random.shuffle(atoms)
        flat = [pos for atom in atoms for pos in atom]
        a_choices = [i for i in range(4, total_tags - 4)]
        if not a_choices:
            continue
        a = random.choice(a_choices)
        b_choices = [j for j in range(a + 4, total_tags)]
        if not b_choices:
            continue
        b = random.choice(b_choices)
        encode_bit_1 = flat[:a]
        encode_bit_0 = flat[a:b]
        neutral_pos  = flat[b:]
        if not (4 <= len(encode_bit_1) <= 9
                and 4 <= len(encode_bit_0) <= 9
                and 1 <= len(neutral_pos) <= 3):
            continue
        if "NOUN" in neutral_pos:
            continue
        if any(tag in neutral_pos for tag in block):
            continue
        det_group = (
            encode_bit_1 if encode_bit_1.count("DET") else encode_bit_0
        )
        if "ADV" in det_group:
            continue
        for forbidden in ("NOUN", "AUX", "PART"):
            for group in (encode_bit_1, encode_bit_0, neutral_pos):
                if "VERB" in group and forbidden in group:
                    break
            else:
                continue
            break
        else:
            return encode_bit_1, encode_bit_0, neutral_pos

In [ ]:
def generate_allowed_pos_transitions(encode_bit_1: list[str], encode_bit_0: list[str], neutral_pos: list[str]) -> set[tuple[str, str]]:
    
    allowed = set()

    def add_symmetric(pos_list):
        for p1 in pos_list:
            for p2 in pos_list:
                if p1 != p2:
                    allowed.add((p1, p2))

    add_symmetric(encode_bit_1)
    add_symmetric(encode_bit_0)
    add_symmetric(neutral_pos)

    for group in [encode_bit_1, encode_bit_0, neutral_pos]:
        for p in group:
            allowed.add((p, p))
    
    for group in (encode_bit_1, encode_bit_0, neutral_pos):
        if "PROPN" in group and "ADJ" in group:
            allowed.discard(("PROPN", "ADJ"))
            allowed.discard(("ADJ", "PROPN"))
            allowed.discard(("NOUN", "PROPN"))
            allowed.discard(("PROPN", "NOUN"))

    return allowed

In [ ]:
def is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions, verbose=True) -> bool:
    disallowed_change_found = False
    for old_token, new_token in zip(stable_tokens, new_stable_tokens):
        if old_token[1] != new_token[1] and (old_token[1], new_token[1]) not in allowed_transitions:
            if verbose:
                print(f"    -> Skipped candidate (POS change for token '{old_token[0]}': {old_token[1]} → {new_token[1]})")
            disallowed_change_found = True
    return disallowed_change_found

def is_pos_shift_detected_noprint(stable_tokens, new_stable_tokens, allowed_transitions) -> bool:
    return is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions, verbose=False)

In [ ]:
def simulate_next_bit_feasibility(model, tokenizer, nlp, context: str, next_bit: str, prompt: str, last_token_id: int, 
                                  candidate_id: int, encode_bit_1: list[str], encode_bit_0: list[str], neutral_pos: list[str], 
                                  allowed_transitions: set[tuple[str,str]], top_k: int = 20) -> bool: 
    inputs = tokenizer(context, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    outputs = model(**inputs)
    logits = outputs.logits[0, -1]
    topk_values, topk_indices = torch.topk(logits, top_k)
    found_pos_match = False
    found_neutral = False

    for i, token_id in enumerate(topk_indices):
        candidate = tokenizer.decode([token_id])
        cleaned = clean_token(candidate)
        if is_new_word(candidate) or is_punctuation(cleaned):
            pos = get_contextual_pos(nlp, context, cleaned)
        else:
            pos = get_contextual_pos_append(nlp, context, cleaned)
        if is_new_word(candidate):
            current_full_text = (context + " " + cleaned).strip()
        else:
            current_full_text = (context + cleaned).strip()
            
        candidate_tensor = torch.tensor([[token_id.item()]], device=device)
        simulated_input_ids = torch.cat([input_ids, candidate_tensor], dim=1)
        stable_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        simulated_full_text = tokenizer.decode(simulated_input_ids[0], skip_special_tokens=True)
        stable_tokens = get_generated_tokens(nlp, prompt, stable_text)
        new_stable_tokens = get_generated_tokens(nlp, prompt, simulated_full_text)
        
        if pos is None:
            print("    [SKIP] POS is None.")
            continue
        if (pos in neutral_pos) and not is_pos_shift_detected_noprint(stable_tokens, new_stable_tokens, allowed_transitions) and (candidate_id != token_id) and is_valid_english_token(cleaned):
            found_neutral = True
        if ((next_bit == "1" and pos in encode_bit_1) or 
            (next_bit == "0" and pos in encode_bit_0)) and \
            not is_pos_shift_detected_noprint(stable_tokens, new_stable_tokens, allowed_transitions) and \
            is_valid_english_token(cleaned) and (candidate_id != token_id):
            found_pos_match = True

    if not found_pos_match and not found_neutral:
        print(f"    -> BOTH Feasibility check failed: BOTH no neutral nor suitable POS in top-{top_k}.")
    return found_pos_match or found_neutral

In [ ]:
def encode_hidden_message(model, tokenizer, nlp, prompt: str, hidden_message: str, encode_bit_1, encode_bit_0, neutral_pos, feas = True) -> tuple[str, bool]:
    torch.cuda.empty_cache()
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    print(f"Initial input tokens: {tokenizer.convert_ids_to_tokens(input_ids[0])}")
    allowed_transitions = generate_allowed_pos_transitions(encode_bit_1, encode_bit_0, neutral_pos)
    exceeded_topk = False
    hidden_token_ids = []
    current_bit_index = 0
    carry_bit: Optional[str] = None
    last_word = ""
    last_word_pos: Optional[str] = None
    last_token_id = input_ids[0][-1].item() 
    ner_counter = 0
    base_top_k = 5
    max_top_k = 1280
    prev_top_k = 0
    outputs = model(input_ids=input_ids)
    next_token_logits = outputs.logits[0, -1, :].to(device)

    while current_bit_index < len(hidden_message) or carry_bit is not None:
        bit_to_encode = carry_bit if carry_bit is not None else hidden_message[current_bit_index]
        bit_info = f"carry '{bit_to_encode}'" if carry_bit is not None else f"'{bit_to_encode}' (index {current_bit_index})"
        print(f"\nEncoding bit {bit_info}...")
        candidate_found = False
        selected_token_id = None
        current_top_k = base_top_k
        current_full_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        stable_tokens = get_generated_tokens(nlp, prompt, current_full_text)

        while not candidate_found and current_top_k <= max_top_k:
            ideal_candidate = None
            acceptable_candidate = None
            higher_six = False
            highest_candidate = None
            highest_candidate_id = None
            highest_candidate_pos = None
            highest_prob = None
            highest_not = None
            topk_values, topk_indices = torch.topk(next_token_logits, current_top_k)
            print(f"Trying candidates {prev_top_k+1}–{current_top_k} of top-{current_top_k}:")
            
            for j, candidate_id in enumerate(topk_indices[prev_top_k:current_top_k], start=prev_top_k):
                candidate_str = tokenizer.decode(candidate_id.item())
                raw_candidate_str = tokenizer.convert_ids_to_tokens(candidate_id.item())
                candidate_prob = topk_values[j].item()
                candidate_tensor = torch.tensor([[candidate_id.item()]], device=device)
                simulated_input_ids = torch.cat([input_ids, candidate_tensor], dim=1)
                simulated_full_text = tokenizer.decode(simulated_input_ids[0], skip_special_tokens=True)
                new_stable_tokens = get_generated_tokens(nlp, prompt, simulated_full_text)
                print(f"  Candidate {j+1}: token='{candidate_str}', raw='{raw_candidate_str}', probability='{candidate_prob}', ID={candidate_id}", end="")
                
                if j == 0:
                    highest_prob = candidate_prob
                    highest_candidate = candidate_str
                    highest_candidate_id = candidate_id.item()

                if re.search(r"Ċ", raw_candidate_str) or re.search(r"<|end_of_text|>", raw_candidate_str):
                    print("    -> Skipped (invalid characters in token)")
                    if j==0:
                        highest_not = True
                    continue
                
                if last_token_id == candidate_id:
                    print("Skip! The same candidate two consecutive times!")
                    continue
                
                if not candidate_str.strip():
                    print("    -> Skipped (empty/whitespace)")
                    if j==0:
                        highest_not = True
                    continue
                
                if last_word_pos == "X" and is_new_word(candidate_str):
                    print("Skip! X POS must be appended, not a new word!")
                    if j==0:
                        highest_not = True
                    continue
                    
                if candidate_id == 20182:
                    print("Invalid token - skip!")
                    continue
                    
                clean_candidate = clean_token(candidate_str)
                if not is_valid_english_token(clean_candidate):
                    print("    -> Skipped (invalid characters in token)")
                    if j==0:
                        highest_not = True
                    continue
                     
                allowed_exceptions = {"'s", "’s", "'t", "’t"}
                punctuation_count = sum(1 for c in clean_candidate if c in string.punctuation)
                if (clean_candidate not in allowed_exceptions and (punctuation_count > 1 or (punctuation_count == 1 and len(clean_candidate.strip(string.punctuation)) > 0))):
                    print("    -> Skipped (punctuation + character or more than one punctuation)")
                    if j==0:
                        highest_not = True
                    continue
                    
                if last_word_pos == "PUNCT" and not is_new_word(candidate_str):
                    print("Skip! After comma, there must be a new word")
                    if j==0:
                        highest_not = True
                    continue
                    
                full_context = tokenizer.decode(input_ids[0], skip_special_tokens=True)
                if is_new_word(candidate_str) or is_punctuation(clean_candidate):
                    pos = get_contextual_pos(nlp, full_context, clean_candidate)
                          
                    if is_all_letters(clean_candidate) and pos == "PUNCT":
                        print(f"Skipping - Classified as PUNCT but contains only letters.")
                        continue

                    if is_all_punct(clean_candidate) and pos != "PUNCT":
                        print(f"Skipping - Classified as {pos} but contains only punctuation.")
                        continue
                    
                    if in_aux_token(clean_candidate):
                        pos = "VERB"
                        print(f"POS Special AUX Case! = {pos}", end="\n")
                    else:
                        print(f"POS = {pos}", end="\n")
                    
                    if ner_counter <= 2 and last_word_pos == "PROPN" and pos == "PROPN" and starts_with_uppercase(last_word) and starts_with_uppercase(clean_candidate) and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        print("Enter NER condition")
                        
                        if acceptable_candidate is None:
                            print(f"  → Recording NER-style acceptable candidate: '{candidate_str}'")
                            acceptable_candidate = (candidate_id.item(), candidate_str, pos)
                        carry_bit = bit_to_encode
                        if j == 0:
                            print("  NER j == 0 – breaking to use as fallback")
                            break
                        else:
                            print("  NER j > 0 – will keep looking for ideal")
                            continue
                    elif ner_counter > 2 and last_word_pos == "PROPN" and pos == "PROPN" and starts_with_uppercase(last_word) and starts_with_uppercase(clean_candidate) and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        print("NER Counter overflow!")
                        continue
                        
                    if pos == "X":
                        print("Skip X token!")
                        continue
                        
                    if candidate_id.item() == 364:
                        print("Skip id 364 token!")
                        continue
                        
                    if current_bit_index + 1 < len(hidden_message): 
                        next_bit = hidden_message[current_bit_index + 1]
                    else:
                        next_bit = None
                    
                    if pos is None:
                        print("    -> Skipped (could not determine POS for appended token)")
                        if j==0:
                            highest_not = True
                            continue
                
                    if j == 0:
                        highest_candidate_pos = pos
                    
                    if bit_to_encode == "0":
                        if (pos in encode_bit_0):
                            candidate_quality = "ideal"
                        elif pos in encode_bit_1:
                            print("    -> Skipped (POS BIT 1 not allowed for bit 0)")
                            if j==0:
                                highest_not = True
                            continue
                        else:
                            candidate_quality = "acceptable"
                    else:
                        if pos in encode_bit_1:
                            candidate_quality = "ideal"
                        elif (pos in encode_bit_0):
                            print("    -> Skipped (POS BIT 0 not allowed for bit 1)")
                            if j==0:
                                highest_not = True
                            continue    
                        else:
                            candidate_quality = "acceptable"

                    if len(new_stable_tokens) < len(stable_tokens):
                        print("    -> Skipped candidate (fewer stable tokens)")
                        if j==0:
                            highest_not = True
                        continue

                    if is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        if j==0:
                            highest_not = True
                        continue
                    
                    if is_punctuation(clean_token(clean_candidate)):
                        text_to_check = (full_context + clean_candidate).strip()
                    else:
                        text_to_check = (full_context + " " + clean_candidate).strip()

                    if acceptable_candidate is None or candidate_quality == "ideal":
                        if feas:
                            if next_bit:
                                feasible = simulate_next_bit_feasibility(model, tokenizer, nlp, text_to_check, next_bit, prompt, last_token_id, candidate_id, 
                                                                         encode_bit_1, encode_bit_0, neutral_pos, allowed_transitions, 20)
                                if not feasible:
                                    print("Skip! No feasible token next")
                                    continue          
                    
                    if candidate_quality == "ideal":
                        ideal_candidate = (candidate_id.item(), candidate_str, pos)
                        break
                    elif acceptable_candidate is None:
                        acceptable_candidate = (candidate_id.item(), candidate_str, pos)

                else:
                    
                    if last_word == "":
                        print("    -> Skipped (no previous word to append to)")
                        if j==0:
                            highest_not = True
                        continue
                        
                    if current_bit_index == 0:
                        print("    -> Skipped (First token after the prompt must be a new word!)")
                        if j==0:
                            highest_not = True
                        continue

                    pos = get_contextual_pos_append(nlp, full_context, candidate_str)

                    if pos is None:
                        print("    -> Skipped (could not determine POS for appended token)")
                        if j==0:
                            highest_not = True
                        continue
                        
                    combined_word = last_word + candidate_str
                    appending = nlp(candidate_str)
                    pos_appending = appending[0].pos_
                    if in_aux_token(clean_candidate):
                        pos_appending = "VERB"
                        print(f"POS Special AUX Case! = {pos_appending}")
                    print(f", POS = {pos_appending}")
                    check_pos_tags(nlp, full_context, candidate_str)

                    if (has_single_pos(combined_word) and (((last_word_pos in encode_bit_0) and (pos in encode_bit_0)) or ((last_word_pos in encode_bit_1) and (pos in encode_bit_1))) and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions)):
                        
                        if ((j == 0)):
                            print(f"1.Ideal/Acceptable candidate token: '{candidate_str}' to form '{combined_word}'. POS of combined word: '{pos}' --> j == 0 or 1 case")
                            candidate_quality = "acceptable"
                            acceptable_candidate = (candidate_id.item(), candidate_str, pos)
                            carry_bit = bit_to_encode
                            candidate_found = False
                            break
                        else:
                            candidate_quality = "acceptable"
                            print(f"1.5Acceptable candidate token: '{candidate_str}' to form '{combined_word}', POS of combined word = {pos}")
                    elif has_single_pos(combined_word) and carry_bit is not None and bit_to_encode == "1" and (last_word_pos in neutral_pos) and pos in encode_bit_1 and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        candidate_quality = "ideal"
                        print(f"2Ideal appending candidate token: '{candidate_str}' to form '{combined_word}'")
                    elif has_single_pos(combined_word) and carry_bit is not None and bit_to_encode == "0" and (last_word_pos in neutral_pos) and pos in encode_bit_0 and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        candidate_quality = "ideal"
                        print(f"3Ideal appending candidate token: '{candidate_str}' to form '{combined_word}'")
                    elif has_single_pos(combined_word) and (last_word_pos in neutral_pos) and (pos in neutral_pos) and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        candidate_quality = "acceptable"
                        print(f"3.5Acceptable appending candidate token: '{candidate_str}' to form '{combined_word}'")
                    elif (candidate_id.item() == 753 or candidate_id.item() == 596 or candidate_id.item() == 956 or candidate_id.item() == 1431):
                        candidate_quality = "acceptable"
                        print(f"4.5Acceptable appending candidate token: '{candidate_str}' to form '{combined_word}'")
                    else:
                        print(f"   -> Skipped: wrong appending POS, pos_appending = {pos_appending} for candidate '{candidate_str}', combined contextual POS: {pos}")
                        if is_pos_shift_detected_noprint(stable_tokens, new_stable_tokens, allowed_transitions):
                            print(f"    -> Skipped: POS change detected!")
                        if not has_single_pos(combined_word):
                            print(f"    -> Skipped: Combined POS longer than 1!")
                            if j==0:
                                highest_not = True
                        continue

                    if len(new_stable_tokens) < len(stable_tokens):
                        print("    -> Skipped candidate (fewer stable tokens)")
                        if j==0:
                            highest_not = True
                        continue

                    if is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        if j==0:
                            highest_not = True
                        continue
                    
                    if candidate_quality == "ideal":
                        ideal_candidate = (candidate_id.item(), candidate_str, pos)
                        break

                    elif acceptable_candidate is None:
                        acceptable_candidate = (candidate_id.item(), candidate_str, pos)
                        
                
            if (
                highest_candidate is not None
                and highest_not is not True
                and highest_candidate_pos != "PUNCT"
                and not is_punctuation(clean_token(highest_candidate))
                and is_new_word(candidate_str)
                and ideal_candidate is not None
                and highest_prob - candidate_prob > 6.1
                and (current_top_k < 11)
                and highest_candidate_pos in neutral_pos
            ):
                print(f"    -> Selected highest probability candidate '{highest_candidate}' (ID {highest_candidate_id}) as acceptable due to probability margin > 6.10")
                acceptable_candidate = (highest_candidate_id, highest_candidate, highest_candidate_pos)
                higher_six = True
                candidate_found = False
            
            if ideal_candidate is not None and not higher_six:
                selected_token_id, token_str, token_pos = ideal_candidate
                if is_new_word(token_str) or is_punctuation(token_str):
                    last_word = token_str
                else:
                    last_word += token_str
                last_word_pos = token_pos
                last_token_id = selected_token_id
                candidate_found = True

                if carry_bit is not None:
                    carry_bit = None
                current_bit_index += 1 
            
                if last_word_pos != "PROPN":
                    ner_counter = 0
                elif starts_with_uppercase(token_str) and token_pos == "PROPN":
                    ner_counter = ner_counter + 1
                    print(f"NER Counter: {ner_counter}, ideal case")
                print(f"    --> Selected new word token (IDEAL): '{token_str}' for bit {bit_to_encode}")
                
                
            elif acceptable_candidate is not None and not candidate_found:
                selected_token_id, token_str, token_pos = acceptable_candidate
                if is_new_word(token_str) or is_punctuation(token_str):
                    last_word = token_str
                else:
                    last_word += token_str
                last_token_id = selected_token_id
                last_word_pos = token_pos
                candidate_found = True
                carry_bit = bit_to_encode
                higher_six = False
                if last_word_pos != "PROPN":
                    ner_counter = 0
                elif starts_with_uppercase(token_str) and token_pos == "PROPN":
                    ner_counter = ner_counter + 1
                    print(f"NER Counter: {ner_counter}, acceptable case")
                print(f"    --> Selected new word token (ACCEPTABLE): '{token_str}' for bit {bit_to_encode} (carrying)")
            
            if not candidate_found:
                print(f"Expanding candidate pool from top-{current_top_k}...")
                prev_top_k = current_top_k
                if current_top_k < 30:
                    current_top_k += 5
                elif current_top_k <= 130:
                    current_top_k += 50
                else:
                    current_top_k += 100
            else:
                prev_top_k = 0
                
            if carry_bit is not None:
                current_top_k = int(current_top_k + 5)
                base_top_k = min(current_top_k, max_top_k)
            else:
                base_top_k = 5

            if current_top_k > 530:
                print("Top-k limit reached. Ending generation early.")
                exceeded_topk = True
                break
                
        if selected_token_id is None:
            print("Warning: no candidate found, ending early.")
            return tokenizer.decode(hidden_token_ids, skip_special_tokens=True), True

        hidden_token_ids.append(selected_token_id)
        token_tensor = torch.tensor([[selected_token_id]], device=device)
        input_ids = torch.cat([input_ids, token_tensor], dim=1)
        outputs = model(input_ids=input_ids)
        next_token_logits = outputs.logits[0, -1, :].to(device)

    stable_text       = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    frozen_prefix     = get_generated_tokens(nlp, prompt, stable_text)
    frozen_prefix_len = len(frozen_prefix)

    max_new            = 40
    top_k              = 1000   
    top_p              = 0.9
    temperature        = 0.8
    repetition_penalty = 1.2

    last_id = input_ids[0, -1].item()

    for i in range(max_new):
        outputs = model(input_ids=input_ids)
        logits  = outputs.logits[0, -1, :]
        for prev in set(input_ids.flatten().tolist()):
            logits[prev] /= repetition_penalty
        logits = logits / temperature
        values, indices = torch.topk(logits, top_k)

        picked_id   = None
        picked_clean = None

        for _, idx in zip(values, indices):
            cand_id = idx.item()
            if cand_id == last_id:
                continue
                
            tok   = tokenizer.decode(cand_id)
            clean = clean_token(tok)
            
            if i == 0 and not (is_new_word(tok) or is_punctuation(clean)):
                continue
            cand_tensor   = torch.tensor([[cand_id]], device=input_ids.device)
            sim_ids       = torch.cat([input_ids, cand_tensor], dim=1)
            sim_text      = tokenizer.decode(sim_ids[0], skip_special_tokens=True)
            new_full      = get_generated_tokens(nlp, prompt, sim_text)
            new_prefix    = new_full[:frozen_prefix_len]

            if is_pos_shift_detected_noprint(frozen_prefix, new_prefix, allowed_transitions):
                continue

            picked_id = cand_id
            picked_clean = clean
            break

        if picked_id is None:
            break

        hidden_token_ids.append(picked_id)
        input_ids = torch.cat([input_ids,
                               torch.tensor([[picked_id]], device=input_ids.device)],
                              dim=1)
        last_id = picked_id

        if picked_clean in {".", "?", "!"}:
            break

    generated_text = tokenizer.decode(hidden_token_ids, skip_special_tokens=True)
    return generated_text, exceeded_topk

In [ ]:
def decode_hidden_message(prompt, text, hidden_message, nlp,
                          encode_bit_1, encode_bit_0, neutral_pos):
    full_text = prompt + text
    doc = nlp(full_text)
    prompt_len = len(prompt)
    exploded = []
    i = 0
    
    while i < len(doc):
        tok = doc[i]
        
        if tok.text.lower() == "can" and i+1 < len(doc) and doc[i+1].text.lower() == "not":
            exploded.append(("cannot", "VERB", tok.idx))
            i += 2
            continue
        
        if i+1 < len(doc) and doc[i+1].text in {"n't", "n’t"}:
            base = tok.text + "n"
            exploded.append((base, tok.pos_, tok.idx))
            exploded.append(("'t", "PART", doc[i+1].idx))
            i += 2
            continue
            
        if tok.text.lower() in {"you", "they"}:
            if i+1 < len(doc) and doc[i+1].text.lower() in {"re"}:
                merged = tok.text + doc[i+1].text
                exploded.append((merged, "VERB", tok.idx))
                i += 2
                continue
        
        exploded.append((tok.text, tok.pos_, tok.idx))
        i += 1

    decoded = ""
    last_pos = None
    last_text = ""
    print("Decoding tokens:")
    for word, pos, idx in exploded:
        if in_aux_token(word):
            pos = "VERB"
        if idx < prompt_len:
            last_pos, last_text = pos, word
            continue
            
        if len(decoded) == len(hidden_message):
            break

        if last_pos == "PROPN" and pos == "PROPN" \
           and last_text[:1].isupper() and word[:1].isupper():
            print(f"Word: '{word}', POS: {pos} -> Skipped (NER‑style)")
        elif word in {"'s", "’s", "'t", "’t"}:
            print(f"Word: '{word}', POS: {pos} -> Skipped (contraction)")
        elif pos in encode_bit_0:
            decoded += "0"
            print(f"Word: '{word}', POS: {pos} -> Decoded as 0")
        elif pos in encode_bit_1:
            decoded += "1"
            print(f"Word: '{word}', POS: {pos} -> Decoded as 1")
        else:
            print(f"Word: '{word}', POS: {pos} -> Skipped (neutral/other)")
        last_pos, last_text = pos, word

    print(f"\nD: {decoded}")
    print(f"H: {hidden_message}")
    if decoded == hidden_message:
        print("Decoded message matches!")
        return decoded, True
    else:
        print("Mismatch.")
        return decoded, False

In [ ]:
def chat_with_gpt(text: str) -> str:
    prompt = (
        f"Rate the following text on a scale from 1 to 100, where 100 means perfect in terms of mistakes, coherence, fluency, grammar, naturalness, and truthfulness, and 1 means it's completely incoherent, full of mistakes or unnatural. As a response just return the number! Text to rate:\n\n\"{text}\"")
    response = openai.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": "You are a helpful and precise text quality evaluator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content.strip()

In [ ]:
# Search for the "optimal" POS group assignment, used in the preliminary experiments described in the paper. 
# The search was performed on separate sets of prompt-hidden message pairs.
def run_experiment(model, tokenizer, nlp, feas, num_runs=5, pos_groups=None):
    best_run = None
    best_avg_score = -1
    best_pos_groups = None

    all_runs = []  

    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        for i in range(1, num_runs + 1):

            if pos_groups is None:
                encode_bit_1, encode_bit_0, neutral_pos = randomize_pos_groups_general()
            else:
                encode_bit_1, encode_bit_0, neutral_pos = pos_groups
                
            allowed_transitions = generate_allowed_pos_transitions(encode_bit_1, encode_bit_0, neutral_pos)

            f.write(f"=== RUN {i} ===\n")
            f.write("Bit 1 POS group: " + ", ".join(encode_bit_1) + "\n")
            f.write("Bit 0 POS group: " + ", ".join(encode_bit_0) + "\n")
            f.write("Neutral POS group: " + ", ".join(neutral_pos) + "\n\n")
            print(f"=== RUN {i} ===\n")
            print("Bit 1 POS group: " + ", ".join(encode_bit_1))
            print("Bit 0 POS group: " + ", ".join(encode_bit_0))
            print("Neutral POS group: " + ", ".join(neutral_pos) + "\n")

            run_ratings = []
            run_exceeded = False

            for j in range(len(PROMPTS)):
                prompt = PROMPTS[j]
                hidden_message = HIDDEN_MESSAGES[j]

                try:
                    generated_text, exceeded_topk = encode_hidden_message( model, tokenizer, nlp, prompt, hidden_message, encode_bit_1, encode_bit_0, neutral_pos, feas = feas)
                    if exceeded_topk:
                        run_exceeded = True
                        
                    decoded_bits, match = decode_hidden_message(prompt, generated_text, hidden_message, nlp, encode_bit_1, encode_bit_0, neutral_pos)

                    if exceeded_topk:
                        rating = 1.0
                    else:
                        rating_str = chat_with_gpt(generated_text)
                        rating = float(rating_str)
                    run_ratings.append(rating)

                    f.write(f"--- Prompt {j + 1} ---\n")
                    match_display = "X" if exceeded_topk else str(match)
                    f.write(f"Match? {match_display}\n")
                    if not match:
                        f.write(f"Decoded bits: {decoded_bits}\n")
                    f.write("Generated Text: " + generated_text + "\n")
                    print("Generated Text: " + generated_text + "\n")
                    if exceeded_topk:
                        f.write("TOPK OVER 1000 - SKIP \n")
                    f.write(f"ChatGPT Rating: {rating}\n\n")
                    f.flush()

                except Exception:
                    trace = traceback.format_exc()
                    f.write(f"[Error in subrun {j + 1} of run {i}]:\n{trace}\n")
                    continue

            if len(run_ratings) == len(PROMPTS):
                avg_rating = sum(run_ratings) / len(PROMPTS)
                f.write(f">>> Summary Ratings for Run {i}: {run_ratings} | Avg: {avg_rating:.2f}\n\n")
                if avg_rating > best_avg_score:
                    best_avg_score = avg_rating
                    best_run = i
                    best_pos_groups = (encode_bit_1, encode_bit_0, neutral_pos)
                all_runs.append((i, avg_rating, run_exceeded, (encode_bit_1, encode_bit_0, neutral_pos)))
            else:
                f.write(f">>> Incomplete ratings for Run {i}: {run_ratings}\n\n")

            f.flush()

        if best_run is not None:
            f.write("\n=== BEST RUN SUMMARY ===\n")
            f.write(f"Best Run: {best_run} with Avg Score: {best_avg_score:.2f}\n")
            f.write("Bit 1 POS group: " + ", ".join(best_pos_groups[0]) + "\n")
            f.write("Bit 0 POS group: " + ", ".join(best_pos_groups[1]) + "\n")
            f.write("Neutral POS group: " + ", ".join(best_pos_groups[2]) + "\n")
            print(f"Best Run: {best_run} with Avg Score: {best_avg_score:.2f}")
        else:
            f.write("\n=== NO COMPLETE RUNS ===\n")
            print("No complete runs were successful.")

        f.write("\n=== ALL RUNS RANKED ===\n")
        sorted_runs = sorted(all_runs, key=lambda x: x[1], reverse=True)
        
        for run_num, avg, exceeded, (g1, g0, gn) in sorted_runs:
            note = " (TOP‑K EXCEEDED)" if exceeded else ""
            f.write(f"Run {run_num}: Avg = {avg:.2f}{note} | 1-bit: {g1} | 0-bit: {g0} | neutral: {gn}\n")
        
        for run_num, avg, exceeded, _ in sorted_runs:
            note = " (TOP‑K)" if exceeded else ""
            print(f"  Run {run_num}: {avg:.2f}{note}")
            
    print(f"Results saved to {RESULTS_FILE}")

In [ ]:
# PROMPTS and ISO 8859-2 Hidden Messages 
PROMPTS = [
    "Galaxies contain billions of stars bound together by gravity, and their shapes range from spirals to ellipticals across the observable universe.",
    "Nebulae are vast clouds of gas and dust where new stars can form under the pull of their own gravity.",
    "Comets travel on long elliptical orbits, releasing gas and dust that form glowing tails as they approach the sun.",
    "Meteors streak across the night sky when small fragments of rock or ice burn up in the atmosphere.",
    "Black holes form when massive stars collapse, creating gravitational fields so strong that not even light can escape.",
    "Satellites orbit Earth to relay communications, monitor weather patterns, and support navigation systems used around the world.",
    "Space telescopes observe distant objects without atmospheric interference, capturing sharper images of galaxies, nebulae, and planets.",
    "Space stations allow astronauts to live and conduct experiments in orbit for extended periods, testing effects of microgravity on the body.",
    "Exoplanets orbit stars beyond our solar system, and astronomers detect them through subtle dips in starlight or gravitational wobbles.",
    "Constellations are patterns of stars that early navigators and astronomers used to mark seasons and chart the night sky.",
    "Cosmic radiation reaches Earth from distant stars and galaxies, and studying it helps scientists understand high energy processes in space.",
    "Volcanoes release molten rock, ash, and gases from beneath Earth's crust, shaping landscapes and occasionally affecting global climate.",
    "Earthquakes occur when built-up stress along fault lines is suddenly released, sending seismic waves through the surrounding rock.",
    "Glaciers form from compacted snow over long periods and slowly carve valleys as they move under their own weight.",
    "Canyons form as rivers and weather gradually erode rock layers, revealing millions of years of geological history in their walls.",
    "Caves develop when water dissolves soft rock like limestone, creating chambers, tunnels, and formations such as stalactites over time.",
    "Fossils preserve traces of ancient organisms in rock, giving scientists evidence about extinct species and how life has changed.",
    "Minerals form naturally through geological processes and are classified by their crystal structure, hardness, and chemical composition.",
    "Tectonic plates shift slowly across Earth's mantle, driving the formation of mountains, ocean trenches, and volcanic activity.",
    "Erosion gradually wears away soil and rock through the action of wind, water, and ice, reshaping landscapes over time.",
    "Sedimentary rock forms as layers of sand, silt, and organic material compress and cement together over long periods.",
    "Groundwater collects in underground layers of rock and soil, supplying wells, springs, and streams during dry periods.",
    "Soil develops from weathered rock and organic matter, and its composition determines which plants can grow successfully in a region.",
    "Hurricanes draw energy from warm ocean water, forming powerful rotating storms with strong winds and heavy rainfall.",
    "Tornadoes form when rotating air within a thunderstorm reaches the ground, producing narrow but intensely destructive wind columns.",
    "Lightning results from the rapid buildup and discharge of electrical energy within storm clouds, heating the surrounding air instantly.",
    "Rainbows appear when sunlight refracts and reflects inside water droplets, separating light into its distinct range of colors.",
    "Snowflakes form as water vapor freezes around tiny particles, and their intricate six-sided patterns depend on temperature and humidity.",
    "Fog forms when air near the ground cools enough for water vapor to condense into tiny suspended droplets.",
    "Humidity measures the amount of water vapor in the air and strongly influences comfort, weather patterns, and cloud formation.",
    "Wind patterns arise from differences in air pressure and temperature, and they help distribute heat and moisture across the planet.",
    "Monsoons bring seasonal shifts in wind direction that deliver heavy rainfall to certain regions for months at a time.",
    "Auroras appear when charged particles from the sun interact with Earth's magnetic field, producing glowing curtains of light near the poles.",
    "Tides rise and fall as the gravitational pull of the moon and sun acts on Earth's oceans throughout the day.",
    "Ocean currents circulate warm and cold water across the globe, influencing climate, marine ecosystems, and weather patterns.",
    "Deep-sea creatures have adapted to survive extreme pressure, cold temperatures, and near total darkness far below the surface.",
    "Whales communicate over long distances using low-frequency sounds and migrate thousands of miles between feeding and breeding grounds.",
    "Sharks have existed for hundreds of millions of years and use keen senses to detect prey across open water.",
    "Octopuses can change color and texture almost instantly, using specialized skin cells to blend into their surroundings.",
    "Plankton drift with ocean currents and form the base of the marine food web, supporting countless larger species.",
    "Kelp forests grow in cold coastal waters and provide shelter and food for a wide variety of marine animals.",
    "Tsunamis form when large volumes of water are displaced by earthquakes or underwater landslides, producing powerful waves that travel far.",
    "Icebergs break away from glaciers or ice shelves and drift through the ocean, with most of their mass hidden underwater.",
    "Estuaries form where rivers meet the sea, creating nutrient-rich habitats that support fish, birds, and other wildlife.",
    "Mangroves grow along tropical coastlines, and their tangled roots protect shorelines from erosion while sheltering young marine life.",
    "Photosynthesis allows plants to convert sunlight, water, and carbon dioxide into energy while releasing oxygen into the atmosphere.",
    "Seeds contain a plant embryo along with stored nutrients, allowing new growth once conditions like moisture and warmth are favorable.",
    "Pollination transfers pollen between flowers, often with help from insects or wind, enabling plants to produce seeds and fruit.",
    "Trees provide oxygen, store carbon, and offer habitat for countless species while stabilizing soil with their extensive root systems.",
    "Mosses lack true roots and instead absorb water directly through their leaves, thriving in shaded and consistently moist environments.",
    "Fungi break down dead organic material, recycling nutrients back into the soil and forming partnerships with many plant roots.",
    "Cacti store water in thick fleshy stems and have adapted spines instead of leaves to survive arid desert conditions.",
    "Orchids produce some of the most varied and intricate flowers in the plant kingdom, often relying on specific pollinators.",
    "Bamboo grows remarkably quickly compared to most plants and is used worldwide for construction, textiles, and food.",
    "Ferns reproduce using spores rather than seeds and have thrived in shaded, humid environments for hundreds of millions of years.",
    "Roots anchor plants in soil while absorbing water and nutrients, and some species store energy reserves underground.",
    "Flowers attract pollinators with color, scent, and shape, playing a central role in how many plants reproduce.",
    "Elephants live in close-knit family groups led by an older female and communicate using low rumbling sounds felt over long distances.",
    "Owls hunt silently at night thanks to specialized feathers that muffle sound, along with exceptional hearing and night vision.",
    "Butterflies undergo complete metamorphosis, transforming from egg to caterpillar to chrysalis before emerging with colorful wings.",
    "Spiders spin silk threads strong enough to capture prey, and different species build webs in distinct shapes and patterns.",
    "Wolves live and hunt in coordinated packs, using vocal calls, body language, and scent to communicate with one another.",
    "Penguins cannot fly but are strong swimmers, using flipper-like wings to move quickly through cold ocean waters.",
    "Bats use echolocation to navigate in darkness, emitting high-frequency calls and listening for echoes to locate insects and obstacles.",
    "Chameleons change color for communication and temperature regulation more often than for camouflage, using specialized skin cells.",
    "Beavers build dams from branches and mud, altering waterways and creating wetland habitats that benefit many other species.",
    "Kangaroos travel by powerful hops and can cover long distances efficiently thanks to their strong hind legs and tails.",
    "Sea turtles migrate thousands of miles across oceans and return to the same beaches where they hatched to lay eggs.",
    "Falcons are known for extraordinary diving speed, folding their wings to plummet toward prey with remarkable precision.",
    "Jellyfish drift through ocean currents using a simple nerve network, and many species use stinging cells to capture prey.",
    "Cheetahs rely on bursts of extreme speed rather than endurance, using their flexible spine and long tail for balance while sprinting.",
    "Ravens are highly intelligent birds capable of solving problems, using tools, and remembering human faces over long periods.",
    "Foxes are adaptable hunters found in forests, grasslands, and cities, relying on keen hearing to locate small prey.",
    "Otters use rocks as tools to crack open shellfish and often float together in groups to rest safely.",
    "Peacocks display elaborate tail feathers during courtship, using color and pattern to attract potential mates.",
    "Flamingos get their pink coloring from pigments in the algae and crustaceans that make up much of their diet.",
    "Koalas feed almost exclusively on eucalyptus leaves and spend much of the day resting to conserve energy.",
    "Pandas rely heavily on bamboo for food, spending many hours each day feeding to meet their nutritional needs.",
    "Camels store fat in their humps rather than water, allowing them to survive for long periods in arid desert climates.",
    "Bison roam grasslands in herds, grazing on native grasses and playing an important role in shaping prairie ecosystems.",
    "The immune system defends the body against pathogens using specialized cells that recognize and eliminate foreign invaders.",
    "The nervous system transmits signals between the brain and body, coordinating movement, sensation, and countless automatic processes.",
    "Muscles contract and relax to produce movement, working in coordinated pairs to control posture, balance, and physical activity.",
    "Bones provide structural support for the body while also storing minerals and producing blood cells within their marrow.",
    "Digestion breaks down food into nutrients the body can absorb, involving organs from the stomach to the small intestine.",
    "Vision relies on light entering the eye and being converted into signals the brain interprets as images.",
    "Hearing depends on tiny structures in the inner ear that convert sound vibrations into electrical signals for the brain.",
    "Skin serves as a protective barrier against infection while also regulating temperature and providing sensory information.",
    "Blood circulation carries oxygen and nutrients throughout the body while removing waste products through a network of vessels.",
    "Memory allows the brain to store and retrieve information, relying on networks of neurons that strengthen with repeated use.",
    "Vaccines train the immune system to recognize specific pathogens, reducing the severity or likelihood of future infection.",
    "Pyramids were constructed as monumental tombs using precisely cut stone blocks, reflecting advanced engineering knowledge of their time.",
    "Castles combined defensive fortifications with living quarters, often built on elevated ground to control surrounding territory.",
    "Ancient empires expanded through conquest and trade, developing administrative systems to govern diverse populations across vast territories.",
    "Explorers charted unfamiliar coastlines and territories, expanding geographic knowledge while establishing new trade routes and contacts.",
    "The Renaissance sparked renewed interest in art, science, and classical learning, transforming European culture over several centuries.",
    "Vikings sailed longships across open seas, establishing trade networks and settlements far beyond their Scandinavian homelands.",
    "Samurai followed a strict code of conduct emphasizing discipline and loyalty while serving as skilled warriors in feudal Japan.",
    "The Silk Road connected distant civilizations, allowing goods, ideas, and technologies to spread across Asia, Europe, and Africa.",
    "Ancient libraries collected written works on scrolls and tablets, preserving knowledge across generations before the spread of printing.",
    "Coinage standardized the exchange of goods by giving currency a consistent, verifiable value backed by governing authorities.",
    "Calendars track the passage of time using astronomical cycles, helping societies coordinate agriculture, festivals, and daily life.",
    "Writing systems allowed information to be recorded and shared beyond spoken communication, transforming how knowledge was preserved.",
    "Aqueducts transported water across long distances using gravity and careful engineering, supplying growing ancient cities.",
    "The Colosseum hosted large public spectacles in ancient Rome, showcasing impressive engineering with its tiered seating and underground chambers.",
    "Knights trained extensively in combat and horsemanship, serving under a code of chivalry that shaped medieval European society.",
    "Mummification preserved bodies through careful drying and wrapping techniques, reflecting beliefs about the afterlife in ancient Egypt.",
    "Feudalism organized medieval society around land ownership, obligating peasants to work land in exchange for protection from nobles.",
    "Nomadic tribes moved seasonally to follow resources like water and grazing land, developing portable shelters and lifestyles.",
    "Trade guilds regulated crafts and commerce in medieval towns, setting standards for training, quality, and pricing.",
    "City-states functioned as independent political units centered on a single city, common in ancient Greece and Renaissance Italy.",
    "Monasteries served as centers of learning and religious practice, often preserving manuscripts through careful copying by hand.",
    "Navigation history traces how sailors used stars, currents, and eventually instruments like the compass to find their way at sea.",
    "Computers process information using binary code, enabling everything from simple calculations to complex simulations and data storage.",
    "The internet connects networks of computers worldwide, allowing information to be shared almost instantly across vast distances.",
    "Batteries store chemical energy and convert it into electricity, powering devices ranging from small electronics to electric vehicles.",
    "Solar panels convert sunlight directly into electricity using semiconductor materials that release electrons when struck by light.",
    "Drones combine lightweight materials, sensors, and remote controls to perform aerial tasks like photography, delivery, and inspection.",
    "Smartphones combine computing, communication, and sensor technology into a single portable device used by billions of people.",
    "3D printing builds objects layer by layer from digital designs, allowing rapid prototyping and customized manufacturing.",
    "Artificial intelligence uses algorithms trained on data to recognize patterns and perform tasks that typically require human reasoning.",
    "Encryption scrambles information using mathematical algorithms so that only authorized parties with the correct key can read it.",
    "GPS relies on a network of satellites to calculate precise locations by measuring the time signals take to arrive.",
    "Semiconductors control the flow of electricity in predictable ways, forming the basis of modern computer chips and electronics.",
    "Lasers produce a focused beam of light through stimulated emission, used in applications from surgery to industrial cutting.",
    "Nanotechnology manipulates materials at an extremely small scale, enabling new properties in medicine, electronics, and manufacturing.",
    "Wireless networks transmit data through radio signals, allowing devices to connect without physical cables.",
    "Virtual reality immerses users in simulated environments using headsets that track movement and adjust the view accordingly.",
    "Microscopes use lenses or electron beams to magnify objects far beyond what the naked eye can perceive.",
    "Radios transmit information by encoding sound onto electromagnetic waves that can be received and decoded at a distance.",
    "Televisions convert broadcast signals into moving images and sound, transforming how information and entertainment reach households.",
    "Refrigeration removes heat from an enclosed space using circulating refrigerants, keeping food and materials cool for longer periods.",
    "Air conditioning cools indoor air by removing heat and humidity, relying on principles of evaporation and compression.",
    "Wind turbines convert the kinetic energy of moving air into electricity using large rotating blades connected to a generator.",
    "Hydroelectric power generates electricity by directing flowing or falling water through turbines connected to generators.",
    "Fiber optics transmit data as pulses of light through thin glass strands, allowing fast and efficient long-distance communication.",
    "Touchscreens detect the position of a finger or stylus through changes in electrical current or light, enabling direct interaction.",
    "Barcodes encode information as patterns of lines that scanners read using reflected light, streamlining inventory and sales.",
    "Sculpture shapes materials like stone, metal, or clay into three-dimensional forms that can be viewed from multiple angles.",
    "Painting applies pigment to a surface using brushes or other tools, allowing artists to express ideas through color and form.",
    "Photography captures light through a lens onto a sensor or film, preserving moments as still images.",
    "Ballet combines precise technique with expressive movement, requiring years of training to master its demanding positions and sequences.",
    "Theater brings stories to life through live performance, combining acting, staging, lighting, and sound before an audience.",
    "Poetry uses rhythm, imagery, and carefully chosen language to convey emotion and meaning in condensed form.",
    "Calligraphy treats handwriting as an art form, using controlled brush or pen strokes to create visually expressive letters.",
    "Pottery shapes clay into functional or decorative objects, which are then hardened through controlled firing at high temperatures.",
    "Origami transforms a flat sheet of paper into intricate three-dimensional shapes using a sequence of precise folds.",
    "Mosaics arrange small pieces of colored glass, stone, or tile into detailed images and patterns on a surface.",
    "Tattoos apply pigment beneath the skin using needles, creating designs that can carry personal, cultural, or symbolic meaning.",
    "Filmmaking combines writing, cinematography, sound, and editing to tell stories through a sequence of moving images.",
    "Animation creates the illusion of movement by displaying a rapid sequence of slightly different images or frames.",
    "Jazz emphasizes improvisation and rhythmic complexity, allowing musicians to reinterpret melodies differently with each performance.",
    "Opera combines orchestral music with singing and staged drama, telling stories through a blend of voice and performance.",
    "Guitars produce sound through vibrating strings, and their tone can be shaped by body shape, materials, and playing technique.",
    "Drums produce sound when a stretched membrane is struck, and different sizes and materials create distinct tones and pitches.",
    "Orchestras bring together instruments from several families, coordinated by a conductor to perform complex musical works.",
    "Choirs blend multiple vocal parts together, relying on timing and harmony to create a unified sound.",
    "Folk music passes musical traditions between generations, often reflecting the history, language, and daily life of a community.",
    "Symphonies are large-scale orchestral compositions typically organized into several contrasting movements that develop musical themes.",
    "Vinyl records store sound as a physical groove that a needle traces to reproduce audio through vibration.",
    "Synthesizers generate and shape electronic sound waves, allowing musicians to create tones not possible with traditional instruments.",
    "Soccer is played between two teams competing to move a ball into the opposing goal using mostly their feet.",
    "Basketball combines speed, coordination, and strategy as players attempt to score by shooting a ball through an elevated hoop.",
    "Tennis players use rackets to hit a ball across a net, relying on precision, footwork, and stamina to outlast opponents.",
    "Swimming relies on coordinated arm and leg movements to move efficiently through water using different competitive strokes.",
    "Skiing uses long attached boards to glide across snow, requiring balance and control while descending varied terrain.",
    "Surfing requires reading ocean waves and using balance to ride their energy across the water's surface.",
    "Rock climbing tests strength, balance, and problem-solving as climbers find routes up natural or artificial rock formations.",
    "Archery combines steady aim with controlled breathing and form to accurately launch arrows toward a distant target.",
    "Gymnastics demands flexibility, strength, and precise control as athletes perform routines on apparatus like bars, beams, and mats.",
    "Marathon running tests endurance over a long fixed distance, requiring careful pacing and sustained physical effort.",
    "Chess challenges players to plan several moves ahead, balancing offense and defense across a grid of sixty-four squares.",
    "Table tennis requires fast reflexes and precise paddle control as players return a lightweight ball across a small table.",
    "Baseball combines pitching, batting, and fielding, with teams alternating turns to score runs around a diamond-shaped field.",
    "Rowing requires coordinated timing and full-body strength as athletes propel a narrow boat using long oars.",
    "Fencing combines quick footwork with precise blade control, as competitors aim to score touches while avoiding their opponent's strikes.",
    "Golf challenges players to hit a ball into a series of holes using as few strokes as possible across varied terrain.",
    "Cycling relies on pedaling power and balance, with riders competing across distances ranging from short sprints to multi-day races.",
    "Diving combines strength and precise body control as athletes perform aerial rotations before entering the water with minimal splash.",
    "Weightlifting tests an athlete's ability to lift maximum loads using techniques that emphasize timing, technique, and full-body coordination.",
    "Curling involves sliding heavy stones across ice toward a target, with teammates sweeping the surface to control the stone's path.",
    "Badminton uses lightweight rackets to hit a feathered shuttlecock across a net, rewarding quick reflexes and agility.",
    "Sailing harnesses wind power through adjustable sails, requiring sailors to read weather and water conditions to navigate efficiently.",
    "Bread forms when flour, water, and often yeast combine, with fermentation and baking producing its characteristic texture and flavor.",
    "Cheese develops through the controlled fermentation and aging of milk, with different techniques producing distinct textures and flavors.",
    "Chocolate comes from cacao beans that are fermented, roasted, and ground before being combined with other ingredients.",
    "Coffee beans are roasted to develop their flavor before being ground and brewed with hot water into a beverage.",
    "Wine develops as yeast ferments the natural sugars in grape juice, with grape variety and aging influencing its final flavor.",
    "Rice farming often relies on flooded fields called paddies, which help control weeds and support the crop's growth cycle.",
    "Beekeeping involves maintaining hives to support honeybee colonies, harvesting honey while helping pollinate surrounding crops.",
    "Fermentation uses microorganisms like yeast and bacteria to convert sugars into alcohol, acids, or gases, preserving and flavoring food.",
    "Spices come from the seeds, bark, roots, or fruits of various plants and have long been traded for their flavor and preservative qualities.",
    "Vegetable gardening involves preparing soil, planting seeds or seedlings, and managing water and sunlight to grow edible crops.",
    "Irrigation supplies water to crops through channels, sprinklers, or drip systems, supporting agriculture in regions with limited rainfall.",
    "Vineyards require careful management of soil, climate, and pruning to produce grapes suited for winemaking.",
    "Dairy farming involves raising cattle or other animals for milk production, which is then processed into various food products.",
    "Fishing has provided food for coastal and river communities for thousands of years, using tools ranging from simple lines to large nets.",
    "Herbal medicine uses plant extracts believed to have therapeutic properties, forming a tradition practiced across many cultures.",
    "Honey is produced when bees collect and process flower nectar, storing it as a concentrated energy source within the hive.",
    "Pasta is typically made from wheat flour and water or eggs, shaped into various forms before being cooked in boiling water.",
    "Sushi combines vinegared rice with ingredients like raw fish, vegetables, or seaweed, prepared according to traditional techniques.",
    "Barbecuing cooks food slowly over indirect heat and smoke, developing deep flavor through prolonged low-temperature cooking.",
    "Pickling preserves food by submerging it in an acidic solution like vinegar, which also develops distinctive tangy flavors.",
    "Sourdough bread relies on a natural culture of wild yeast and bacteria to leaven the dough instead of commercial yeast.",
    "Ice cream forms when a sweetened cream mixture is churned while freezing, incorporating air to create a smooth texture.",
    "Maple syrup is made by boiling down sap collected from maple trees until it reduces into a thick, sweet liquid.",
    "Mountains form through tectonic uplift, volcanic activity, or erosion, creating some of the most dramatic landscapes on Earth.",
    "Islands form through volcanic activity, coral growth, or the erosion of larger landmasses, and their isolation often produces unique ecosystems.",
    "Peninsulas extend from a larger landmass into surrounding water, often shaping regional climate, trade, and settlement patterns.",
    "Plateaus are elevated, relatively flat areas formed by processes like volcanic activity, erosion, or tectonic uplift.",
    "Fjords form when glaciers carve deep valleys that are later filled by seawater, creating steep-walled coastal inlets.",
    "Savannas feature grasslands scattered with trees, supporting large herds of grazing animals adapted to seasonal rainfall.",
    "Rainforests support extraordinary biodiversity, with dense, layered vegetation shaped by consistent rainfall and warm temperatures.",
    "Tundra environments feature low temperatures and permanently frozen subsoil, limiting plant life to hardy, low-growing species.",
    "Wetlands filter water and provide habitat for many species, forming a transitional zone between land and water.",
    "Volcanic islands form when underwater eruptions build up enough material to rise above the ocean's surface over time.",
    "Archipelagos consist of clusters of islands, often formed through related volcanic or tectonic activity in a single region.",
    "Oases form where underground water reaches the surface in arid regions, supporting vegetation and human settlement in the desert.",
    "Geometry studies the properties of shapes, sizes, and spatial relationships, forming a foundation for fields like architecture and engineering.",
    "Algebra uses symbols and letters to represent numbers and relationships, allowing general problems to be solved systematically.",
    "Statistics collects, analyzes, and interprets data, helping researchers identify patterns and draw conclusions from uncertain information.",
    "Probability measures the likelihood of different outcomes, providing a mathematical framework for reasoning about uncertainty and chance.",
    "Fractals display repeating patterns at every scale, appearing in natural forms like coastlines, snowflakes, and plant growth.",
    "Number theory examines the properties and relationships of whole numbers, including prime numbers and their distribution.",
    "Calculus studies continuous change through concepts like derivatives and integrals, underlying much of modern physics and engineering.",
    "Cryptography uses mathematical techniques to secure information, allowing messages to be encoded so only intended recipients can read them.",
    "Gravity attracts objects with mass toward one another, shaping everything from falling objects to the orbits of planets.",
    "Magnetism arises from the movement of electric charges, producing forces that attract or repel certain materials.",
    "Electricity results from the movement of electric charge through a conductor, powering countless modern technologies and devices.",
    "Chemical reactions occur when substances interact to form new products, often releasing or absorbing energy in the process.",
    "Thermodynamics studies how heat and energy transfer between systems, governing processes from engines to weather patterns.",
    "Optics examines how light behaves as it reflects, refracts, and bends, explaining phenomena from lenses to rainbows.",
    "Atoms consist of a nucleus surrounded by electrons, and their arrangement determines the properties of every chemical element.",
    "Sound waves travel as vibrations through a medium like air or water, carried by regions of compression and rarefaction.",
    "Radioactivity occurs when unstable atomic nuclei release energy and particles as they decay into more stable forms.",
    "Crystals form when atoms or molecules arrange themselves into a repeating, orderly structure as a substance solidifies.",
    "Superconductors conduct electricity with zero resistance below a certain temperature, enabling powerful magnets and efficient energy transfer.",
    "Plasma forms when gas becomes ionized, freeing electrons and creating a distinct state of matter found in stars and lightning.",
    "Trains move along fixed rails, allowing efficient transport of passengers and heavy freight across long distances.",
    "Ships rely on buoyancy and displaced water to stay afloat, carrying goods and passengers across oceans and waterways.",
    "Airplanes generate lift as air moves faster over curved wings, allowing controlled flight through the atmosphere.",
    "Submarines control their depth by adjusting buoyancy through ballast tanks, allowing them to travel underwater for extended periods.",
    "Hot air balloons rise as heated air inside the envelope becomes less dense than the surrounding cooler atmosphere.",
    "Elevators use motors, cables, and counterweights to move passengers and goods vertically between floors of a building.",
    "Highways connect distant regions with wide, high-capacity roads designed to move large volumes of traffic efficiently.",
    "Canals are constructed waterways that connect bodies of water, historically used to transport goods and enable irrigation.",
    "Bridges span obstacles like rivers or valleys, using designs such as arches, trusses, or suspension cables to distribute weight.",
    "Subways move large numbers of passengers through underground tunnels, reducing congestion on streets above.",
    "Alphabets use a set of symbols to represent the sounds of a language, allowing spoken words to be written down.",
    "Sign language uses hand shapes, movement, and facial expression to communicate, forming complete languages with their own grammar.",
    "Translation converts text or speech from one language into another while attempting to preserve meaning, tone, and context.",
    "Braille represents letters and numbers as patterns of raised dots, allowing text to be read through touch.",
    "Morse code represents letters and numbers using sequences of short and long signals, historically transmitted by sound or light.",
    "Storytelling passes knowledge, values, and history between generations through spoken, written, or performed narratives.",
    "Proverbs condense practical wisdom into short, memorable phrases that are passed down within a culture over time.",
    "Etymology traces how words originate and change meaning over time, revealing connections between different languages.",
    "Skyscrapers rely on steel frameworks and reinforced foundations to support their height while resisting wind and seismic forces.",
    "Lighthouses use powerful rotating lights to warn ships of dangerous coastlines, reefs, and shallow waters at night.",
    "Domes distribute weight evenly across a curved surface, allowing large open spaces without the need for internal supports.",
    "Cathedrals were built with soaring ceilings and large stained glass windows, showcasing advanced engineering for their era.",
    "Windmills convert the kinetic energy of wind into mechanical power, historically used for grinding grain or pumping water.",
    "Dams hold back flowing water to create reservoirs, supporting irrigation, flood control, and hydroelectric power generation.",
    "Towers are built tall and narrow relative to their base, often serving purposes like observation, communication, or defense.",
    "Amphitheaters use tiered seating arranged around a central stage or arena, designed to project sound to large audiences.",
    "Clocks measure the passage of time using consistent mechanical or electronic oscillations, from swinging pendulums to vibrating crystals.",
    "Mirrors reflect light off a smooth, polished surface, producing an image that appears to be behind the glass.",
    "Umbrellas use a collapsible frame covered in waterproof fabric to shield people from rain or strong sunlight.",
    "Candles burn a wick surrounded by wax, which melts and is drawn upward to fuel a steady flame.",
    "Soap molecules bind to both water and oil, allowing them to lift dirt and grease away from surfaces during washing.",
    "Glass forms when silica-based materials are heated to extremely high temperatures and then cooled without crystallizing.",
    "Furniture is designed to balance function and comfort, using materials and joinery suited to its intended use.",
    "Textiles are created by weaving, knitting, or felting fibers together, producing materials used for clothing and countless other purposes."
]

HIDDEN_MESSAGES = [
    "010001110110000101101100", # Gal (0x47, 0x61, 0x6C) - Starts with 010
    "111100010100010101100010", # ńEb (0xF1, 0x45, 0x62) - Starts with 11
    "011000110011000001101101", # c0m (0x63, 0x30, 0x6D) - Starts with 011
    "010011010110010101010100", # MeT (0x4D, 0x65, 0x54) - Starts with 010
    "011000100110110000110100", # bl4 (0x62, 0x6C, 0x34) - Starts with 011
    "010100110110000101110100", # Sat (0x53, 0x61, 0x74) - Starts with 010
    "010101000110010110110011", # Teł (0x54, 0x65, 0xB3) - Starts with 010
    "011100110101010000110100", # sT4 (0x73, 0x54, 0x34) - Starts with 011
    "110010100111100001001111", # ĘxO (0xCA, 0x78, 0x4F) - Starts with 11
    "010000110110111101101110", # Con (0x43, 0x6F, 0x6E) - Starts with 010
    "110001100100111101110011", # ĆOs (0xC6, 0x4F, 0x73) - Starts with 11
    "010101100110111100110001", # Vo1 (0x56, 0x6F, 0x31) - Starts with 010
    "110010100100000101110010", # ĘAr (0xCA, 0x41, 0x72) - Starts with 11
    "011001110110110000110100", # gl4 (0x67, 0x6C, 0x34) - Starts with 011
    "010000110110000101101110", # Can (0x43, 0x61, 0x6E) - Starts with 010
    "011000111010000101110110", # cĄv (0x63, 0xA1, 0x76) - Starts with 011
    "011001100011000001010011", # f0S (0x66, 0x30, 0x53) - Starts with 011
    "010011010110100101101110", # Min (0x4D, 0x69, 0x6E) - Starts with 010
    "011101000011001101100011", # t3c (0x74, 0x33, 0x63) - Starts with 011
    "010001010111001001101111", # Ero (0x45, 0x72, 0x6F) - Starts with 010
    "101010010100010101100100", # ŠEd (0xA9, 0x45, 0x64) - Starts with 10
    "001110010101001001101111", # 9Ro (0x39, 0x52, 0x6F) - Starts with 001
    "101010010110111101101001", # Šoi (0xA9, 0x6F, 0x69) - Starts with 10
    "011010000111010101110010", # hur (0x68, 0x75, 0x72) - Starts with 011
    "010101000110111101110010", # Tor (0x54, 0x6F, 0x72) - Starts with 010
    "101000110110100101100111", # Łig (0xA3, 0x69, 0x67) - Starts with 10
    "011100100011010001001001", # r4I (0x72, 0x34, 0x49) - Starts with 011
    "101010010110111001001111", # ŠnO (0xA9, 0x6E, 0x4F) - Starts with 10
    "011001100110111100111001", # fo9 (0x66, 0x6F, 0x39) - Starts with 011
    "010010000111010101101101", # Hum (0x48, 0x75, 0x6D) - Starts with 010
    "011101110110100111010001", # wiŃ (0x77, 0x69, 0xD1) - Starts with 011
    "010011010011000001101110", # M0n (0x4D, 0x30, 0x6E) - Starts with 010
    "101000010111010101010010", # ĄuR (0xA1, 0x75, 0x52) - Starts with 10
    "011101000011000101100100", # t1d (0x74, 0x31, 0x64) - Starts with 011
    "010011110110001101100101", # Oce (0x4F, 0x63, 0x65) - Starts with 010
    "110011110110010101100101", # Ďee (0xCF, 0x65, 0x65) - Starts with 11
    "011101110110100000110100", # wh4 (0x77, 0x68, 0x34) - Starts with 011
    "101010010100100001100001", # ŠHa (0xA9, 0x48, 0x61) - Starts with 10
    "011011110110001100110111", # oc7 (0x6F, 0x63, 0x37) - Starts with 011
    "010100000110110001100001", # Pla (0x50, 0x6C, 0x61) - Starts with 010
    "011010111100110001101100", # kĚl (0x6B, 0xCC, 0x6C) - Starts with 011
    "001101110111001101010101", # 7sU (0x37, 0x73, 0x55) - Starts with 001
    "010010010100001101100101", # ICe (0x49, 0x43, 0x65) - Starts with 010
    "011001010111001100110111", # es7 (0x65, 0x73, 0x37) - Starts with 011
    "010011010110000101101110", # Man (0x4D, 0x61, 0x6E) - Starts with 010
    "011100000110100011010011", # phÓ (0x70, 0x68, 0xD3) - Starts with 011
    "001101010110010101000101", # 5eE (0x35, 0x65, 0x45) - Starts with 001
    "010100000110111101001100", # PoL (0x50, 0x6F, 0x4C) - Starts with 010
    "011101000111001000110011", # tr3 (0x74, 0x72, 0x33) - Starts with 011
    "010011010110111101110011", # Mos (0x4D, 0x6F, 0x73) - Starts with 010
    "010001100111010111110001", # Fuń (0x46, 0x75, 0xF1) - Starts with 010
    "011000110011010001100011", # c4c (0x63, 0x34, 0x63) - Starts with 011
    "110100110111001001000011", # ÓrC (0xD3, 0x72, 0x43) - Starts with 11
    "011000100011010001101101", # b4m (0x62, 0x34, 0x6D) - Starts with 011
    "010001100110010101110010", # Fer (0x46, 0x65, 0x72) - Starts with 010
    "011100101111001101001111", # róO (0x72, 0xF3, 0x4F) - Starts with 011
    "011001100100110000110000", # fL0 (0x66, 0x4C, 0x30) - Starts with 011
    "110011000110110001000101", # ĚlE (0xCC, 0x6C, 0x45) - Starts with 11
    "010011110111011101101100", # Owl (0x4F, 0x77, 0x6C) - Starts with 010
    "010000100111010110111011", # Buť (0x42, 0x75, 0xBB) - Starts with 010
    "011100110111000000110001", # sp1 (0x73, 0x70, 0x31) - Starts with 011
    "010101110110111101101100", # Wol (0x57, 0x6F, 0x6C) - Starts with 010
    "011100000011001101101110", # p3n (0x70, 0x33, 0x6E) - Starts with 011
    "010000100110000101110100", # Bat (0x42, 0x61, 0x74) - Starts with 010
    "111001100100100001100001", # ćHa (0xE6, 0x48, 0x61) - Starts with 11
    "011000100110010100110100", # be4 (0x62, 0x65, 0x34) - Starts with 011
    "010010110110000101101110", # Kan (0x4B, 0x61, 0x6E) - Starts with 010
    "011100110110010100110100", # se4 (0x73, 0x65, 0x34) - Starts with 011
    "010001100110000101101100", # Fal (0x46, 0x61, 0x6C) - Starts with 010
    "010010101110101001101100", # Jęl (0x4A, 0xEA, 0x6C) - Starts with 010
    "011000110110100000110011", # ch3 (0x63, 0x68, 0x33) - Starts with 011
    "110110000110000101110110", # Řav (0xD8, 0x61, 0x76) - Starts with 11
    "011001100011000001111000", # f0x (0x66, 0x30, 0x78) - Starts with 011
    "010011110111010001110100", # Ott (0x4F, 0x74, 0x74) - Starts with 010
    "010100000110010100110100", # Pe4 (0x50, 0x65, 0x34) - Starts with 010
    "010001100100110001100001", # FLa (0x46, 0x4C, 0x61) - Starts with 010
    "011010110110111100110100", # ko4 (0x6B, 0x6F, 0x34) - Starts with 011
    "010100000110000101101110", # Pan (0x50, 0x61, 0x6E) - Starts with 010
    "011000111010000101101101", # cĄm (0x63, 0xA1, 0x6D) - Starts with 011
    "011000100011000101010011", # b1S (0x62, 0x31, 0x53) - Starts with 011
    "010010010110110101001101", # ImM (0x49, 0x6D, 0x4D) - Starts with 010
    "011011100011001101110010", # n3r (0x6E, 0x33, 0x72) - Starts with 011
    "010011010111010101110011", # Mus (0x4D, 0x75, 0x73) - Starts with 010
    "011000101111001101001110", # bóN (0x62, 0xF3, 0x4E) - Starts with 011
    "011001000110100100111001", # di9 (0x64, 0x69, 0x39) - Starts with 011
    "011101100110100100110101", # vi5 (0x76, 0x69, 0x35) - Starts with 011
    "010010000110010101100001", # Hea (0x48, 0x65, 0x61) - Starts with 010
    "101101100110101101001001", # śkI (0xB6, 0x6B, 0x49) - Starts with 10
    "011000100110110000110000", # bl0 (0x62, 0x6C, 0x30) - Starts with 011
    "010011010100010101101101", # MEm (0x4D, 0x45, 0x6D) - Starts with 010
    "011101100011010001100011", # v4c (0x76, 0x34, 0x63) - Starts with 011
    "010100000111100101110010", # Pyr (0x50, 0x79, 0x72) - Starts with 010
    "111001100110000101010011", # ćaS (0xE6, 0x61, 0x53) - Starts with 11
    "001100110110110101110000", # 3mp (0x33, 0x6D, 0x70) - Starts with 001
    "110011000111100001110000", # Ěxp (0xCC, 0x78, 0x70) - Starts with 11
    "011100100011001101101110", # r3n (0x72, 0x33, 0x6E) - Starts with 011
    "010101100110100101101011", # Vik (0x56, 0x69, 0x6B) - Starts with 010
    "101001100100000101101101", # ŚAm (0xA6, 0x41, 0x6D) - Starts with 10
    "010100110110100100110001", # Si1 (0x53, 0x69, 0x31) - Starts with 010
    "101000110110100101100010", # Łib (0xA3, 0x69, 0x62) - Starts with 10
    "011000110110111100110001", # co1 (0x63, 0x6F, 0x31) - Starts with 011
    "010000110110000101101100", # Cal (0x43, 0x61, 0x6C) - Starts with 010
    "011101111111100001001001", # wřI (0x77, 0xF8, 0x49) - Starts with 011
    "001101000111000101110101", # 4qu (0x34, 0x71, 0x75) - Starts with 001
    "110010000110111101101100", # Čol (0xC8, 0x6F, 0x6C) - Starts with 11
    "011010110110111000110001", # kn1 (0x6B, 0x6E, 0x31) - Starts with 011
    "010011010111010101101101", # Mum (0x4D, 0x75, 0x6D) - Starts with 010
    "010001100100010101110101", # FEu (0x46, 0x45, 0x75) - Starts with 010
    "011011100011000001101101", # n0m (0x6E, 0x30, 0x6D) - Starts with 011
    "010101000111001001100001", # Tra (0x54, 0x72, 0x61) - Starts with 010
    "010000110110100110101011", # CiŤ (0x43, 0x69, 0xAB) - Starts with 010
    "011011010011000001101110", # m0n (0x6D, 0x30, 0x6E) - Starts with 011
    "110100100110000101010110", # ŇaV (0xD2, 0x61, 0x56) - Starts with 11
    "011000110011000001101101", # c0m (0x63, 0x30, 0x6D) - Starts with 011
    "010010010110111001110100", # Int (0x49, 0x6E, 0x74) - Starts with 010
    "010000101011000101110100", # Bąt (0x42, 0xB1, 0x74) - Starts with 010
    "011100110100111100110001", # sO1 (0x73, 0x4F, 0x31) - Starts with 011
    "110011110111001001101111", # Ďro (0xCF, 0x72, 0x6F) - Starts with 11
    "011100110110110100110100", # sm4 (0x73, 0x6D, 0x34) - Starts with 011
    "010100000111001001101001", # Pri (0x50, 0x72, 0x69) - Starts with 010
    "101000010111001001110100", # Ąrt (0xA1, 0x72, 0x74) - Starts with 10
    "001100110110111001000011", # 3nC (0x33, 0x6E, 0x43) - Starts with 001
    "010001110101000001110011", # GPs (0x47, 0x50, 0x73) - Starts with 010
    "011100110011001101101101", # s3m (0x73, 0x33, 0x6D) - Starts with 011
    "010011000110000101110011", # Las (0x4C, 0x61, 0x73) - Starts with 010
    "011011101011000101001110", # nąN (0x6E, 0xB1, 0x4E) - Starts with 011
    "011101110011000101110010", # w1r (0x77, 0x31, 0x72) - Starts with 011
    "010101100100100101110010", # VIr (0x56, 0x49, 0x72) - Starts with 010
    "011011010011000101100011", # m1c (0x6D, 0x31, 0x63) - Starts with 011
    "010100100110000101100100", # Rad (0x52, 0x61, 0x64) - Starts with 010
    "101110110110010101001100", # ťeL (0xBB, 0x65, 0x4C) - Starts with 10
    "011100100011001101000110", # r3F (0x72, 0x33, 0x46) - Starts with 011
    "101000010110100101110010", # Ąir (0xA1, 0x69, 0x72) - Starts with 10
    "011101110011000101101110", # w1n (0x77, 0x31, 0x6E) - Starts with 011
    "010010000111100101100100", # Hyd (0x48, 0x79, 0x64) - Starts with 010
    "011001100110100101000010", # fiB (0x66, 0x69, 0x42) - Starts with 011
    "001101110100111101110101", # 7Ou (0x37, 0x4F, 0x75) - Starts with 001
    "010000100110000101110010", # Bar (0x42, 0x61, 0x72) - Starts with 010
    "001101010110001101110101", # 5cu (0x35, 0x63, 0x75) - Starts with 001
    "010100000110000101101001", # Pai (0x50, 0x61, 0x69) - Starts with 010
    "011100000100100011010011", # pHÓ (0x70, 0x48, 0xD3) - Starts with 011
    "011000100110000100110001", # ba1 (0x62, 0x61, 0x31) - Starts with 011
    "101010110100100001100101", # ŤHe (0xAB, 0x48, 0x65) - Starts with 10
    "011100000110111100110011", # po3 (0x70, 0x6F, 0x33) - Starts with 011
    "010000110110000101101100", # Cal (0x43, 0x61, 0x6C) - Starts with 010
    "011100000100111110111011", # pOť (0x70, 0x4F, 0xBB) - Starts with 011
    "001100000101001001101001", # 0Ri (0x30, 0x52, 0x69) - Starts with 001
    "010011010100111101110011", # MOs (0x4D, 0x4F, 0x73) - Starts with 010
    "011101000110000100110111", # ta7 (0x74, 0x61, 0x37) - Starts with 011
    "011001100110100110100011", # fiŁ (0x66, 0x69, 0xA3) - Starts with 011
    "011000010100111000110001", # aN1 (0x61, 0x4E, 0x31) - Starts with 011
    "011010100011010001111010", # j4z (0x6A, 0x34, 0x7A) - Starts with 011
    "010011110111000001100101", # Ope (0x4F, 0x70, 0x65) - Starts with 010
    "011001110101010101101001", # gUi (0x67, 0x55, 0x69) - Starts with 011
    "011001000101001001110101", # dRu (0x64, 0x52, 0x75) - Starts with 011
    "110100110101001001100011", # ÓRc (0xD3, 0x52, 0x63) - Starts with 11
    "011000110110100000110000", # ch0 (0x63, 0x68, 0x30) - Starts with 011
    "010001100110111101101100", # Fol (0x46, 0x6F, 0x6C) - Starts with 010
    "101010010111100101101101", # Šym (0xA9, 0x79, 0x6D) - Starts with 10
    "011101100011000101001110", # v1N (0x76, 0x31, 0x4E) - Starts with 011
    "101010010111100101001110", # ŠyN (0xA9, 0x79, 0x4E) - Starts with 10
    "011100110011000001100011", # s0c (0x73, 0x30, 0x63) - Starts with 011
    "010000100110000101110011", # Bas (0x42, 0x61, 0x73) - Starts with 010
    "101010110110010101101110", # Ťen (0xAB, 0x65, 0x6E) - Starts with 10
    "001101010101011101101001", # 5Wi (0x35, 0x57, 0x69) - Starts with 001
    "101010010100101101101001", # ŠKi (0xA9, 0x4B, 0x69) - Starts with 10
    "001101010111010101110010", # 5ur (0x35, 0x75, 0x72) - Starts with 001
    "010100100110111101100011", # Roc (0x52, 0x6F, 0x63) - Starts with 010
    "011000010111001011000110", # arĆ (0x61, 0x72, 0xC6) - Starts with 011
    "001110010101100101101101", # 9Ym (0x39, 0x59, 0x6D) - Starts with 001
    "010011010110000101010010", # MaR (0x4D, 0x61, 0x52) - Starts with 010
    "011000110110100000110011", # ch3 (0x63, 0x68, 0x33) - Starts with 011
    "010101000110000101100010", # Tab (0x54, 0x61, 0x62) - Starts with 010
    "011000101010000101010011", # bĄS (0x62, 0xA1, 0x53) - Starts with 011
    "011100100011000001110111", # r0w (0x72, 0x30, 0x77) - Starts with 011
    "010001100100010101101110", # FEn (0x46, 0x45, 0x6E) - Starts with 010
    "011001110110111100110001", # go1 (0x67, 0x6F, 0x31) - Starts with 011
    "010000110111100101100011", # Cyc (0x43, 0x79, 0x63) - Starts with 010
    "110011110110100101010110", # ĎiV (0xCF, 0x69, 0x56) - Starts with 11
    "011101110011001101101001", # w3i (0x77, 0x33, 0x69) - Starts with 011
    "110001100101010101110010", # ĆUr (0xC6, 0x55, 0x72) - Starts with 11
    "011000100011010001100100", # b4d (0x62, 0x34, 0x64) - Starts with 011
    "101101100100000101101001", # śAi (0xB6, 0x41, 0x69) - Starts with 10
    "001110000101001001100101", # 8Re (0x38, 0x52, 0x65) - Starts with 001
    "110010000110100001000101", # ČhE (0xC8, 0x68, 0x45) - Starts with 11
    "011000110110100000110000", # ch0 (0x63, 0x68, 0x30) - Starts with 011
    "010000110110111101100110", # Cof (0x43, 0x6F, 0x66) - Starts with 010
    "010101110110100111010010", # WiŇ (0x57, 0x69, 0xD2) - Starts with 010
    "010100100011000101100011", # R1c (0x52, 0x31, 0x63) - Starts with 010
    "010000100100010101100101", # BEe (0x42, 0x45, 0x65) - Starts with 010
    "011001100011001101110010", # f3r (0x66, 0x33, 0x72) - Starts with 011
    "010100110111000001101001", # Spi (0x53, 0x70, 0x69) - Starts with 010
    "011101101110110001000111", # věG (0x76, 0xEC, 0x47) - Starts with 011
    "001100010111001001110010", # 1rr (0x31, 0x72, 0x72) - Starts with 001
    "010101100110100101001110", # ViN (0x56, 0x69, 0x4E) - Starts with 010
    "011001000110000100110001", # da1 (0x64, 0x61, 0x31) - Starts with 011
    "010001100110100101110011", # Fis (0x46, 0x69, 0x73) - Starts with 010
    "011010001100101001110010", # hĘr (0x68, 0xCA, 0x72) - Starts with 011
    "011010000011000001101110", # h0n (0x68, 0x30, 0x6E) - Starts with 011
    "010100000100000101110011", # PAs (0x50, 0x41, 0x73) - Starts with 010
    "011100110111010100110101", # su5 (0x73, 0x75, 0x35) - Starts with 011
    "010000100110000101110010", # Bar (0x42, 0x61, 0x72) - Starts with 010
    "010100000110100111100110", # Pić (0x50, 0x69, 0xE6) - Starts with 010
    "001101010110111101110101", # 5ou (0x35, 0x6F, 0x75) - Starts with 001
    "010010010110001101000101", # IcE (0x49, 0x63, 0x45) - Starts with 010
    "011011010011010001110000", # m4p (0x6D, 0x34, 0x70) - Starts with 011
    "010011010110111101110101", # Mou (0x4D, 0x6F, 0x75) - Starts with 010
    "011010011010100101001100", # iŠL (0x69, 0xA9, 0x4C) - Starts with 011
    "011100000011001101001110", # p3N (0x70, 0x33, 0x4E) - Starts with 011
    "010100000100110001100001", # PLa (0x50, 0x4C, 0x61) - Starts with 010
    "011001100110101000110000", # fj0 (0x66, 0x6A, 0x30) - Starts with 011
    "010100110110000101110110", # Sav (0x53, 0x61, 0x76) - Starts with 010
    "010100101011000101101001", # Rąi (0x52, 0xB1, 0x69) - Starts with 010
    "001101110111010101101110", # 7un (0x37, 0x75, 0x6E) - Starts with 001
    "010101110110010101110100", # Wet (0x57, 0x65, 0x74) - Starts with 010
    "011101100110111100110001", # vo1 (0x76, 0x6F, 0x31) - Starts with 011
    "010000010111001001100011", # Arc (0x41, 0x72, 0x63) - Starts with 010
    "011011110110000110100110", # oaŚ (0x6F, 0x61, 0xA6) - Starts with 011
    "011001110011001101101111", # g3o (0x67, 0x33, 0x6F) - Starts with 011
    "101000010110110001100111", # Ąlg (0xA1, 0x6C, 0x67) - Starts with 10
    "011100110111010000110100", # st4 (0x73, 0x74, 0x34) - Starts with 011
    "010100000111001001101111", # Pro (0x50, 0x72, 0x6F) - Starts with 010
    "011001100111001010100001", # frĄ (0x66, 0x72, 0xA1) - Starts with 011
    "011011100101010101101101", # nUm (0x6E, 0x55, 0x6D) - Starts with 011
    "110010000110000101001100", # ČaL (0xC8, 0x61, 0x4C) - Starts with 11
    "011000110111001001111001", # cry (0x63, 0x72, 0x79) - Starts with 011
    "010001110111001001100001", # Gra (0x47, 0x72, 0x61) - Starts with 010
    "011011011010000101100111", # mĄg (0x6D, 0xA1, 0x67) - Starts with 011
    "001100110100110001100101", # 3Le (0x33, 0x4C, 0x65) - Starts with 001
    "110010000110100001100101", # Čhe (0xC8, 0x68, 0x65) - Starts with 11
    "011101000110100000110011", # th3 (0x74, 0x68, 0x33) - Starts with 011
    "010011110111000001110100", # Opt (0x4F, 0x70, 0x74) - Starts with 010
    "010000010111010011010011", # AtÓ (0x41, 0x74, 0xD3) - Starts with 010
    "011100110011000001010101", # s0U (0x73, 0x30, 0x55) - Starts with 011
    "110110000110000101100100", # Řad (0xD8, 0x61, 0x64) - Starts with 11
    "011000110111001001111001", # cry (0x63, 0x72, 0x79) - Starts with 011
    "010100110111010101110000", # Sup (0x53, 0x75, 0x70) - Starts with 010
    "011100001010001101000001", # pŁA (0x70, 0xA3, 0x41) - Starts with 011
    "001101110111001001100001", # 7ra (0x37, 0x72, 0x61) - Starts with 001
    "101010010110100001101001", # Šhi (0xA9, 0x68, 0x69) - Starts with 10
    "011000010011000101110010", # a1r (0x61, 0x31, 0x72) - Starts with 011
    "010100110111010101100010", # Sub (0x53, 0x75, 0x62) - Starts with 010
    "011010000100111110111011", # hOť (0x68, 0x4F, 0xBB) - Starts with 011
    "011001010100110000110011", # eL3 (0x65, 0x4C, 0x33) - Starts with 011
    "010010000110100101100111", # Hig (0x48, 0x69, 0x67) - Starts with 010
    "011000110011010001101110", # c4n (0x63, 0x34, 0x6E) - Starts with 011
    "010000100111001001101001", # Bri (0x42, 0x72, 0x69) - Starts with 010
    "101101100111010101000010", # śuB (0xB6, 0x75, 0x42) - Starts with 10
    "001101000110110001010000", # 4lP (0x34, 0x6C, 0x50) - Starts with 001
    "101001100110100101100111", # Śig (0xA6, 0x69, 0x67) - Starts with 10
    "011101000111001000110100", # tr4 (0x74, 0x72, 0x34) - Starts with 011
    "010000100111001001100001", # Bra (0x42, 0x72, 0x61) - Starts with 010
    "011011011101001101010010", # mÓR (0x6D, 0xD3, 0x52) - Starts with 011
    "010100110111010000110000", # St0 (0x53, 0x74, 0x30) - Starts with 010
    "010100000111001001001111", # PrO (0x50, 0x72, 0x4F) - Starts with 010
    "011001010011011101111001", # e7y (0x65, 0x37, 0x79) - Starts with 011
    "010100110110101101111001", # Sky (0x53, 0x6B, 0x79) - Starts with 010
    "101100110100100101100111", # łIg (0xB3, 0x49, 0x67) - Starts with 10
    "011001000011000001001101", # d0M (0x64, 0x30, 0x4D) - Starts with 011
    "110001100110000101110100", # Ćat (0xC6, 0x61, 0x74) - Starts with 11
    "011101110011000101101110", # w1n (0x77, 0x31, 0x6E) - Starts with 011
    "010001000110000101101101", # Dam (0x44, 0x61, 0x6D) - Starts with 010
    "011101001111001101010111", # tóW (0x74, 0xF3, 0x57) - Starts with 011
    "001101000110110101010000", # 4mP (0x34, 0x6D, 0x50) - Starts with 001
    "110010000110110001001111", # ČlO (0xC8, 0x6C, 0x4F) - Starts with 11
    "011011010011000101110010", # m1r (0x6D, 0x31, 0x72) - Starts with 011
    "010101010110110101100010", # Umb (0x55, 0x6D, 0x62) - Starts with 010
    "011000111010000101001110", # cĄN (0x63, 0xA1, 0x4E) - Starts with 011
    "001101010100111101100001", # 5Oa (0x35, 0x4F, 0x61) - Starts with 001
    "010001110110110001000001", # GlA (0x47, 0x6C, 0x41) - Starts with 010
    "010001100111010101110010", # Fur (0x46, 0x75, 0x72) - Starts with 010
    "101010110110010101111000" # Ťex (0xAB, 0x65, 0x78) - Starts with 10
]

In [ ]:
encode_bit_1 = ["VERB", "ADP", "SCONJ", "DET", "PRON", "ADJ", "PUNCT"]
encode_bit_0 = ["ADV", "AUX", "PART", "PROPN", "NOUN", "CCONJ", "NUM"]
neutral_pos = ["INTJ", "SYM"]

In [ ]:
import json
import pandas as pd

In [ ]:
def generate_cover_text(model, tokenizer, prompt, max_new=40,
                         top_k=1000, top_p=0.9, temperature=0.8, repetition_penalty=1.2,
                         safety_cap=500):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    generated_ids = []
    last_id = input_ids[0, -1].item()
    steps = 0

    while True:
        steps += 1
        if steps > safety_cap:
            print(f"[WARN] generate_cover_text: reached safety_cap={safety_cap} "
                  f"sampling steps without a sentence-ending punctuation mark; "
                  f"returning the text generated so far (prompt: '{prompt[:40]}...').")
            break

        outputs = model(input_ids=input_ids)
        logits = outputs.logits[0, -1, :].clone()
        for prev in set(input_ids.flatten().tolist()):
            logits[prev] /= repetition_penalty
        logits = logits / temperature

        topk_vals, topk_idx = torch.topk(logits, top_k)
        probs = torch.softmax(topk_vals, dim=-1)
        sorted_probs, sorted_order = torch.sort(probs, descending=True)
        cumulative = torch.cumsum(sorted_probs, dim=-1)
        cutoff = max((cumulative <= top_p).sum().item() + 1, 1)
        chosen = torch.multinomial(sorted_probs[:cutoff], 1).item()
        picked_id = topk_idx[sorted_order[chosen]].item()

        if picked_id == last_id:
            continue

        generated_ids.append(picked_id)
        input_ids = torch.cat([input_ids, torch.tensor([[picked_id]], device=device)], dim=1)
        last_id = picked_id

        token_str = clean_token(tokenizer.decode(picked_id))
        long_enough = len(generated_ids) >= max_new
        if token_str in {".", "?", "!"} and long_enough:
            break

    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [ ]:
def build_paired_corpus(model, tokenizer, nlp, prompts, hidden_messages,
                         encode_bit_1, encode_bit_0, neutral_pos,
                         feas=True, out_path="generated_corpus.jsonl",
                         mismatch_path="decode_mismatches.jsonl"):

    import io
    from contextlib import redirect_stdout

    records = []
    skipped = 0
    mismatches = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for idx, (prompt, msg) in enumerate(zip(prompts, hidden_messages)):
            try:
                stego_text, exceeded = encode_hidden_message(
                    model, tokenizer, nlp, prompt, msg,
                    encode_bit_1, encode_bit_0, neutral_pos, feas=feas)
                if exceeded:
                    skipped += 1
                    continue 

                decode_buf = io.StringIO()
                with redirect_stdout(decode_buf):
                    decoded_msg, decode_ok = decode_hidden_message(
                        prompt, stego_text, msg, nlp,
                        encode_bit_1, encode_bit_0, neutral_pos)
                if not decode_ok:
                    mismatches += 1
                    print(f"  [DECODE MISMATCH] prompt {idx}: expected={msg} decoded={decoded_msg}")
                    with open(mismatch_path, "a", encoding="utf-8") as mf:
                        mf.write(json.dumps({
                            "prompt": prompt, "hidden_message": msg,
                            "decoded_message": decoded_msg, "stego_text": stego_text,
                        }, ensure_ascii=False) + "\n")

                cover_text = generate_cover_text(model, tokenizer, prompt)

                for label, text in [("stego", stego_text), ("cover", cover_text)]:
                    rec = {"prompt": prompt, "text": text, "label": label, "score": chat_with_gpt(stego_text) if label == "stego" else None,
                           "hidden_message": msg if label == "stego" else None,
                           "decode_match": decode_ok if label == "stego" else None}
                    records.append(rec)
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    f.flush()

                if (idx + 1) % 25 == 0:
                    print(f"  ... {idx + 1}/{len(prompts)} prompts processed")

            except Exception as e:
                print(f"[WARN] Error for prompt {idx} ('{prompt[:30]}...'): {e}")
                skipped += 1
                continue

    print(f"Skipped (top-k exceeded / error): {skipped} / {len(prompts)}")
    print(f"Decoding errors: {mismatches} (saved to {mismatch_path})")
    return pd.DataFrame(records)

In [ ]:
df = build_paired_corpus(
    model, tokenizer, nlp, PROMPTS, HIDDEN_MESSAGES,
    encode_bit_1, encode_bit_0, neutral_pos,
    out_path="generated_corpus.jsonl",
)
print(df["label"].value_counts())

In [ ]:
LENGTH_PROMPTS = [
    "Books provide windows into countless worlds and ideas, serving as enduring sources of knowledge, entertainment, and inspiration.",
    "Artificial intelligence is transforming industries by automating tasks, analyzing vast data sets, and enabling more personalized experiences.",
    "Psychology is the scientific study of the mind and behavior, seeking to understand why individuals think, feel, and act the way they do.",
    "Plants need sunlight and water to grow.",
    "Regular exercise is important for maintaining good health.",
]

SECRET_SENTENCES = [
    "Warsaw University of Technology produces world-class engineers.",
    "Artificial intelligence is one of the most rapidly evolving technologies.",
    "Meta Llama 3.1 is a multilingual large language model.",
]

BIT_LENGTHS = list(range(16, 112 + 1, 16))
LENGTH_RESULTS_FILE = "length_experiment_results.jsonl"


def text_to_bits(text):
    return "".join(format(b, "08b") for b in text.encode("iso-8859-2"))


SENTENCE_BITS = [text_to_bits(s) for s in SECRET_SENTENCES]

In [ ]:
import gc
import time


def run_length_experiment(model, tokenizer, nlp, encode_bit_1, encode_bit_0, neutral_pos,
                          feas=True, out_path=LENGTH_RESULTS_FILE):
    failed_combos = set()
    records = []
    with open(out_path, "w", encoding="utf-8") as f:
        for n_bits in BIT_LENGTHS:
            for p_idx, prompt in enumerate(LENGTH_PROMPTS):
                for s_idx, sent_bits in enumerate(SENTENCE_BITS):
                    if (p_idx, s_idx) in failed_combos:
                        continue

                    msg = sent_bits[:n_bits]
                    t0 = time.time()
                    status, rating, match, text = "ok", None, None, None

                    try:
                        text, exceeded = encode_hidden_message(
                            model, tokenizer, nlp, prompt, msg,
                            encode_bit_1, encode_bit_0, neutral_pos, feas=feas)
                        if exceeded:
                            status = "topk_exceeded"
                            rating = 1.0
                        else:
                            _, match = decode_hidden_message(
                                prompt, text, msg, nlp,
                                encode_bit_1, encode_bit_0, neutral_pos)
                            rating = float(chat_with_gpt(text))
                    except torch.cuda.OutOfMemoryError:
                        status = "oom"
                        failed_combos.add((p_idx, s_idx))
                        torch.cuda.empty_cache()
                        gc.collect()
                    except Exception as e:
                        status = "error"
                        print(f"[WARN] Error for P{p_idx} S{s_idx}, {n_bits} bits: {e}")

                    if status == "topk_exceeded":
                        failed_combos.add((p_idx, s_idx))

                    rec = {
                        "n_bits": n_bits, "n_chars": n_bits // 8,
                        "prompt_idx": p_idx, "sentence_idx": s_idx,
                        "hidden_message": msg, "status": status,
                        "rating": rating, "decode_match": match,
                        "generated_text": text, "elapsed_s": round(time.time() - t0, 1),
                    }
                    records.append(rec)
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    f.flush()
                    print(f"{n_bits} bits, P{p_idx} S{s_idx}: {status}, rating={rating}, match={match}")

    return pd.DataFrame(records)

In [ ]:
df_length = run_length_experiment(model, tokenizer, nlp, encode_bit_1, encode_bit_0, neutral_pos)
print(df_length["status"].value_counts())